# RevXcel - Customer Segmentation using K-Means Clustering
## Aligned with Project Documentation (Section 3.4)

This notebook implements K-Means clustering for customer segmentation as mentioned in the project documentation.


## Step 1: Import Libraries and Load Data


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
import os
warnings.filterwarnings('ignore')

# Fix for Windows threadpoolctl/OpenBLAS issues
# Set environment variables BEFORE importing sklearn to prevent threading issues
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

# Monkey patch threadpool_info to avoid OpenBLAS errors
try:
    import threadpoolctl
    original_threadpool_info = threadpoolctl.threadpool_info

    def safe_threadpool_info():
        """Safe wrapper that returns empty list on OpenBLAS errors"""
        try:
            return original_threadpool_info()
        except (AttributeError, OSError, TypeError) as e:
            # Return empty list to prevent sklearn from crashing
            return []

    threadpoolctl.threadpool_info = safe_threadpool_info

    # Also patch sklearn's internal use
    try:
        from sklearn.utils import fixes
        if hasattr(fixes, 'threadpool_info'):
            fixes.threadpool_info = safe_threadpool_info
    except Exception:
        pass

    print("✓ Applied threadpoolctl safety patch")
except Exception as e:
    print(f"Note: Threadpoolctl patch skipped ({type(e).__name__})")

print("✓ Libraries imported")


✓ Applied threadpoolctl safety patch
✓ Libraries imported


In [15]:
# Load processed customer data with RFM features
# According to doc.txt Section 3.4, we need CLV, RFM scores, frequency, channel, product category
try:
    # First try to load customer_metrics.csv
    df = pd.read_csv('/content/sample_data/customer_metrics.csv')
    print(f"✓ Loaded customer_metrics.csv: {df.shape}")

    # Check if RFM features are missing - if so, load from sales_data_processed.csv
    required_rfm_cols = ['Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score']
    missing_rfm = [col for col in required_rfm_cols if col not in df.columns]

    if missing_rfm:
        print(f"⚠ Missing RFM features in customer_metrics.csv: {missing_rfm}")
        print("Loading from sales_data_processed.csv to get complete RFM data...")

        df_processed = pd.read_csv('/content/sample_data/sales_data_processed.csv')
        if 'Customer Name' in df_processed.columns:
            # Aggregate by customer, taking first value for RFM metrics (they should be consistent per customer)
            df_agg = df_processed.groupby('Customer Name').agg({
                'Total Sales': 'sum',
                'CLV': 'first',
                'Purchase_Frequency': 'first',
                'RFM_Segment': 'first',
                'Customer Type': 'first',
                'City': 'first',
                'Recency': 'first',
                'Frequency': 'first',
                'Monetary': 'first',
                'R_Score': 'first',
                'F_Score': 'first',
                'M_Score': 'first',
                'Channel': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown',  # Most common channel
            }).reset_index()

            # Merge with existing customer_metrics to preserve any additional data
            df = df.merge(df_agg[['Customer Name'] + missing_rfm + (['Channel'] if 'Channel' not in df.columns else [])],
                         on='Customer Name', how='left', suffixes=('', '_new'))

            # Fill missing values from merged data
            for col in missing_rfm:
                if f'{col}_new' in df.columns:
                    df[col] = df[col].fillna(df[f'{col}_new'])
                    df = df.drop(columns=[f'{col}_new'])

            if 'Channel_new' in df.columns:
                df['Channel'] = df['Channel'].fillna(df['Channel_new'])
                df = df.drop(columns=['Channel_new'])

            print(f"✓ Enhanced customer_metrics with RFM features: {df.shape}")
        else:
            print("⚠ Could not load RFM features from sales_data_processed.csv")
except FileNotFoundError:
    # If customer_metrics.csv doesn't exist, create from processed data
    try:
        df_processed = pd.read_csv('sales_data_processed.csv')
        if 'Customer Name' in df_processed.columns:
            df = df_processed.groupby('Customer Name').agg({
                'Total Sales': 'sum',
                'CLV': 'first',
                'Purchase_Frequency': 'first',
                'RFM_Segment': 'first',
                'Customer Type': 'first',
                'City': 'first',
                'Recency': 'first',
                'Frequency': 'first',
                'Monetary': 'first',
                'R_Score': 'first',
                'F_Score': 'first',
                'M_Score': 'first',
                'Channel': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown',
            }).reset_index()
            print(f"✓ Created customer metrics from processed data: {df.shape}")
        else:
            print("❌ Error: Customer Name column not found")
            raise FileNotFoundError
    except FileNotFoundError:
        print("❌ Error: No processed data found. Please run preprocess.ipynb first")
        raise

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()


✓ Created customer metrics from processed data: (26, 14)

Dataset shape: (26, 14)
Columns: ['Customer Name', 'Total Sales', 'CLV', 'Purchase_Frequency', 'RFM_Segment', 'Customer Type', 'City', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Channel']

First few rows:


,Customer Name,Total Sales,CLV,Purchase_Frequency,RFM_Segment,Customer Type,City,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Channel
0,Abdullah Alomary,3688205,7.384497e+05,373.408935,Potential Loyalists,Loyal Customer,AlUla,0,1865,3688205,2,3,3,OnLine
1,Abeer Mohamed,5121551,1.024871e+06,530.090262,Champions,Loyal Customer,Riyadh,0,2649,5121551,2,5,5,OnLine
2,Ahmed Ezzat,3679941,7.363914e+05,377.006437,Potential Loyalists,Loyal Customer,Riyadh,0,1884,3679941,2,3,2,OnLine
3,Ahmed Yousef,3528748,7.061363e+05,362.198329,Potential Loyalists,Loyal Customer,Riyadh,0,1810,3528748,2,2,2,OnLine
4,Amal Abdelfatah,4850377,9.711389e+05,487.734137,Loyal Customers,Loyal Customer,Hafar Al Batin,0,2436,4850377,2,4,5,OnLine


## Step 2: Prepare Features for Clustering


In [16]:
# Select features for clustering (as per doc.txt Section 3.4: CLV, RFM scores, frequency, channel, product category)
# Priority: Use RFM scores (R_Score, F_Score, M_Score) as they are normalized, plus CLV and Purchase_Frequency
feature_columns = []

# Core features (as per documentation)
if 'CLV' in df.columns:
    feature_columns.append('CLV')
if 'Purchase_Frequency' in df.columns:
    feature_columns.append('Purchase_Frequency')

# RFM Scores (preferred over raw RFM values as they are normalized)
if 'R_Score' in df.columns:
    feature_columns.append('R_Score')
if 'F_Score' in df.columns:
    feature_columns.append('F_Score')
if 'M_Score' in df.columns:
    feature_columns.append('M_Score')

# If RFM scores are not available, use raw RFM values
if not any(col in feature_columns for col in ['R_Score', 'F_Score', 'M_Score']):
    print("⚠ RFM Scores not found, using raw RFM values instead")
    if 'Recency' in df.columns:
        feature_columns.append('Recency')
    if 'Frequency' in df.columns:
        feature_columns.append('Frequency')
    if 'Monetary' in df.columns:
        feature_columns.append('Monetary')

# Ensure we have at least 2 features for clustering
if len(feature_columns) < 2:
    print("⚠ Warning: Less than 2 features available. Adding available numeric features...")
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if col not in feature_columns and col not in ['Total Sales']:  # Exclude Total Sales as it's redundant with CLV
            feature_columns.append(col)
            if len(feature_columns) >= 3:
                break

print(f"Selected features for clustering: {feature_columns}")

# Prepare data
X = df[feature_columns].copy()

# Check for missing values
missing_counts = X.isnull().sum()
if missing_counts.sum() > 0:
    print(f"\n⚠ Missing values found:")
    print(missing_counts[missing_counts > 0])
    print("Dropping rows with missing values...")
    X = X.dropna()
    print(f"Rows after dropping NaN: {len(X)} (from {len(df)})")

print(f"\nData shape after removing NaN: {X.shape}")
print(f"\nFeature statistics:")
print(X.describe())


Selected features for clustering: ['CLV', 'Purchase_Frequency', 'R_Score', 'F_Score', 'M_Score']

Data shape after removing NaN: (26, 5)

Feature statistics:
                CLV  Purchase_Frequency    R_Score    F_Score    M_Score
count  2.600000e+01           26.000000  26.000000  26.000000  26.000000
mean   7.597723e+05          385.013235   1.807692   2.923077   2.923077
std    2.623415e+05          133.938532   0.401918   1.467599   1.467599
min    3.870000e+03            1.000000   1.000000   1.000000   1.000000
25%    6.998636e+05          352.886460   2.000000   2.000000   2.000000
50%    7.414957e+05          377.906931   2.000000   3.000000   3.000000
75%    9.471203e+05          482.580027   2.000000   4.000000   4.000000
max    1.217447e+06          608.666575   2.000000   5.000000   5.000000


In [ ]:
# Standardize features (important for K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_columns, index=X.index)

print("✓ Features standardized")
print(f"Scaled data shape: {X_scaled_df.shape}")
X_scaled_df.head()


✓ Features standardized
Scaled data shape: (26, 5)


,CLV,Purchase_Frequency,R_Score,F_Score,M_Score
0,-0.082888,-0.088355,0.48795,0.053452,0.053452
1,1.030524,1.104612,0.48795,1.443211,1.443211
2,-0.090889,-0.060964,0.48795,0.053452,-0.641427
3,-0.208500,-0.173712,0.48795,-0.641427,-0.641427
4,0.821649,0.782114,0.48795,0.748331,1.443211


: 

## Step 3: Determine Optimal Number of Clusters (Elbow Method)


In [18]:
# Elbow method to find optimal k
# Use regular KMeans with n_jobs=1 to avoid threading issues
print("Finding optimal number of clusters using Elbow Method and Silhouette Score...")

inertias = []
silhouette_scores = []
# Limit K_range to ensure k < n_samples (required for K-Means)
# For small datasets, limit to reasonable range
max_k = min(10, len(X_scaled) // 2)  # At least 2 samples per cluster
K_range = range(2, max_k + 1)

print(f"Testing k values from {min(K_range)} to {max(K_range)} (dataset size: {len(X_scaled)})")

for k in K_range:
    try:
        # Use regular KMeans with single thread to avoid OpenBLAS issues
        kmeans = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10,  # Multiple initializations for better results
            max_iter=300,
            # n_jobs=1  # Single thread to avoid threading issues - REMOVED n_jobs
        )
        kmeans.fit(X_scaled)
        labels = kmeans.labels_
        inertias.append(kmeans.inertia_)

        # Calculate silhouette score (requires at least 2 clusters and 2 samples per cluster)
        if k >= 2 and len(np.unique(labels)) > 1:
            sil_score = silhouette_score(X_scaled, labels)
            silhouette_scores.append(sil_score)
        else:
            silhouette_scores.append(-1)  # Invalid clustering

        print(f"✓ k={k}: Inertia={inertias[-1]:.2f}, Silhouette={silhouette_scores[-1]:.3f}")
    except Exception as e:
        print(f"❌ Error with k={k}: {e}")
        # Fallback: calculate approximate inertia
        approx_inertia = len(X_scaled) * np.var(X_scaled, axis=0).sum() / k
        inertias.append(approx_inertia)
        silhouette_scores.append(-1)  # Invalid score

# Plot elbow curve (outside threadpool context)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
ax1.plot(K_range, inertias, 'bo-')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia (Within-cluster sum of squares)')
ax1.set_title('Elbow Method for Optimal k')
ax1.grid(True, alpha=0.3)

# Silhouette score plot
ax2.plot(K_range, silhouette_scores, 'ro-')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score for Optimal k')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find optimal k (highest silhouette score, ignoring invalid scores)
valid_silhouette_scores = [s for s in silhouette_scores if s >= 0]
if len(valid_silhouette_scores) > 0:
    valid_indices = [i for i, s in enumerate(silhouette_scores) if s >= 0]
    best_idx = valid_indices[np.argmax(valid_silhouette_scores)]
    optimal_k = list(K_range)[best_idx]
    print(f"\nOptimal number of clusters: {optimal_k} (Silhouette Score: {max(valid_silhouette_scores):.3f})")
else:
    # Fallback: use elbow method if silhouette scores are all invalid
    print("\n⚠ All silhouette scores invalid, using elbow method...")
    # Calculate rate of change in inertia
    if len(inertias) > 1:
        inertia_changes = [inertias[i] - inertias[i+1] for i in range(len(inertias)-1)]
        # Find where the rate of change decreases significantly (elbow)
        if len(inertia_changes) > 1:
            change_ratios = [inertia_changes[i] / inertia_changes[i+1] if inertia_changes[i+1] > 0 else 0
                           for i in range(len(inertia_changes)-1)]
            if len(change_ratios) > 0:
                # Find first significant drop in change ratio
                elbow_idx = next((i for i, ratio in enumerate(change_ratios) if ratio > 1.5), 0)
                optimal_k = list(K_range)[min(elbow_idx + 1, len(K_range) - 1)]
            else:
                optimal_k = 3  # Default
        else:
            optimal_k = 3  # Default
    else:
        optimal_k = 2  # Minimum
    print(f"Selected k={optimal_k} using elbow method")

: 

: 

## Step 4: Apply K-Means Clustering


In [ ]:
# Apply K-Means with optimal k
# Use regular KMeans with single thread to avoid threading issues
kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10,
    max_iter=300,
    # n_jobs=1  # Single thread to avoid OpenBLAS issues - REMOVED n_jobs
)
cluster_labels = kmeans.fit_predict(X_scaled)

# Add cluster labels to dataframe
df_clustered = df.loc[X.index].copy()
df_clustered['Cluster'] = cluster_labels
df_clustered['Cluster'] = df_clustered['Cluster'].astype(str)

print(f"✓ K-Means clustering completed with {optimal_k} clusters")
print(f"\nCluster distribution:")
print(df_clustered['Cluster'].value_counts().sort_index())
print(f"\nTotal customers clustered: {len(df_clustered)}")

✓ K-Means clustering completed with 4 clusters

Cluster distribution:
Cluster
0     2
1    13
2     9
3     2
Name: count, dtype: int64

Total customers clustered: 26


## Step 5: Analyze Cluster Characteristics and Save Results


In [ ]:
# Analyze cluster characteristics
# Build aggregation dictionary based on available columns
agg_dict = {}
numeric_cols = df_clustered.select_dtypes(include=[np.number]).columns.tolist()

# Always include CLV and Total Sales
if 'CLV' in numeric_cols:
    agg_dict['CLV'] = ['mean', 'std', 'count']
if 'Total Sales' in numeric_cols:
    agg_dict['Total Sales'] = 'sum'
if 'Purchase_Frequency' in numeric_cols:
    agg_dict['Purchase_Frequency'] = 'mean'

# Add RFM features if available
for col in ['Recency', 'Frequency', 'Monetary']:
    if col in numeric_cols:
        agg_dict[col] = 'mean'

# Add RFM scores if available
for col in ['R_Score', 'F_Score', 'M_Score']:
    if col in numeric_cols:
        agg_dict[col] = 'mean'

if len(agg_dict) == 0:
    print("⚠ Warning: No numeric columns found for cluster analysis")
    cluster_summary = pd.DataFrame()
else:
    cluster_summary = df_clustered.groupby('Cluster').agg(agg_dict).round(2)

print("Cluster Characteristics:")
print("=" * 70)
if not cluster_summary.empty:
    print(cluster_summary)
else:
    print("No cluster summary available")

# Assign cluster names based on characteristics
def assign_cluster_name(cluster_id, cluster_data, df_ref):
    """Assign meaningful names to clusters based on their characteristics"""
    try:
        # Get CLV (primary metric)
        if 'CLV' in cluster_data.index.get_level_values(0) if hasattr(cluster_data.index, 'get_level_values') else []:
            clv = cluster_data['CLV']['mean'] if isinstance(cluster_data['CLV'], pd.Series) else cluster_data.get(('CLV', 'mean'), cluster_data.get('CLV', 0))
        else:
            clv = cluster_data.get('CLV', df_ref['CLV'].mean() if 'CLV' in df_ref.columns else 0)

        # Get Frequency (if available)
        if 'Frequency' in df_ref.columns:
            freq = cluster_data.get('Frequency', df_ref['Frequency'].mean())
            freq_median = df_ref['Frequency'].quantile(0.5)
        else:
            freq = 0
            freq_median = 0

        # Get Recency (if available)
        if 'Recency' in df_ref.columns:
            recency = cluster_data.get('Recency', df_ref['Recency'].mean())
            recency_median = df_ref['Recency'].quantile(0.5)
            recency_75 = df_ref['Recency'].quantile(0.75)
        else:
            recency = 0
            recency_median = 0
            recency_75 = 0

        clv_median = df_ref['CLV'].quantile(0.5) if 'CLV' in df_ref.columns else 0
        clv_75 = df_ref['CLV'].quantile(0.75) if 'CLV' in df_ref.columns else 0

        # Assign segment based on RFM logic
        if clv > clv_75 and freq > freq_median:
            return 'Champions'
        elif clv > clv_median and recency < recency_median:
            return 'Loyal Customers'
        elif freq > freq_median:
            return 'Potential Loyalists'
        elif recency > recency_75:
            return 'At Risk'
        else:
            return 'Lost Customers'
    except Exception as e:
        # Fallback naming
        return f'Segment {cluster_id}'

# Create cluster name mapping
cluster_names = {}
if not cluster_summary.empty:
    for cluster_id in df_clustered['Cluster'].unique():
        try:
            cluster_data = cluster_summary.loc[cluster_id]
            cluster_names[cluster_id] = assign_cluster_name(cluster_id, cluster_data, df_clustered)
        except Exception as e:
            cluster_names[cluster_id] = f'Segment {cluster_id}'
else:
    # Simple naming if no summary available
    for i, cluster_id in enumerate(sorted(df_clustered['Cluster'].unique())):
        cluster_names[cluster_id] = f'Segment {i+1}'

df_clustered['Cluster_Name'] = df_clustered['Cluster'].map(cluster_names)
print(f"\nCluster Names:")
for cluster_id, name in cluster_names.items():
    count = (df_clustered['Cluster'] == cluster_id).sum()
    print(f"  Cluster {cluster_id}: {name} ({count} customers)")

# Visualize clusters
# Use available features for visualization
x_feature = 'CLV' if 'CLV' in df_clustered.columns else feature_columns[0] if len(feature_columns) > 0 else 'Total Sales'
y_feature = None

# Try to find a good y-feature
for feat in ['Frequency', 'Purchase_Frequency', 'Monetary', 'F_Score']:
    if feat in df_clustered.columns:
        y_feature = feat
        break

if y_feature is None and len(feature_columns) > 1:
    y_feature = feature_columns[1]

if x_feature and y_feature and x_feature in df_clustered.columns and y_feature in df_clustered.columns:
    hover_cols = ['Customer Name', 'Cluster_Name']
    for col in ['Recency', 'Monetary', 'Total Sales']:
        if col in df_clustered.columns:
            hover_cols.append(col)

    size_col = 'Monetary' if 'Monetary' in df_clustered.columns else 'Total Sales' if 'Total Sales' in df_clustered.columns else None

    fig = px.scatter(
        df_clustered,
        x=x_feature,
        y=y_feature,
        color='Cluster_Name',
        size=size_col,
        hover_data=hover_cols,
        title=f'Customer Segmentation - {x_feature} vs {y_feature}',
        labels={x_feature: x_feature, y_feature: y_feature}
    )
    fig.update_layout(height=500)
    fig.show()
    print(f"\n✓ Visualization created: {x_feature} vs {y_feature}")
else:
    print(f"⚠ Cannot create visualization: missing required features (x: {x_feature}, y: {y_feature})")

# Save results
import joblib
df_clustered.to_csv('customers_clustered.csv', index=False)
joblib.dump(kmeans, 'kmeans_model.pkl')
joblib.dump(scaler, 'kmeans_scaler.pkl')

print("\n✓ Saved clustered data and models")
print("=" * 70)


Cluster Characteristics:
               CLV                 Total Sales Purchase_Frequency Recency  \
              mean       std count         sum               mean    mean   
Cluster                                                                     
0        716095.34  37454.18     2     7149190             353.28    2.00   
1        725468.72  45174.66    13    47118063             366.29    0.31   
2        986959.01  92812.41     9    44375211             504.39    0.33   
3          4082.48    300.49     2        9529               1.26   77.50   

        Frequency    Monetary R_Score F_Score M_Score  
             mean        mean    mean    mean    mean  
Cluster                                                
0         1763.50  3574595.00    1.00    1.50    2.00  
1         1830.00  3624466.38    2.00    2.31    2.23  
2         2519.78  4930579.00    1.89    4.56    4.56  
3            1.50     4764.50    1.00    1.00    1.00  

Cluster Names:
  Cluster 1: Segment 1 (13 


✓ Visualization created: CLV vs Frequency

✓ Saved clustered data and models
